### Ingest drivers.json file
1. Read the file using spark dataframe API
2. Define and enforce schema (preserve the nested structure)
3. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
4. Write bronze delta table    

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructField,StructType,StringType,DateType

In [0]:
name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType([
       StructField('driverId', StringType()),
       StructField('dateOfBirth', DateType()),
       StructField('name', name_schema),
       StructField('nationality', StringType()),
       StructField('url', StringType())
    ])


In [0]:
drivers_df = spark.read.format('json').schema(drivers_schema).option('mode','FAILFAST').load(source_file)

In [0]:
display(drivers_df)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))